# 4. Monte Carlo Control
### Learns an Optimal Policy via Generalized Policy Iteration (GPI)

**Monte Carlo Control** adapts Generalized Policy Iteration to model-free learning. Instead of state values $V(s)$, it evaluates **state-action values $Q(s, a)$** because without a transition model, $V(s)$ alone is insufficient to select greedy actions.

#### Key Principles:
1. **Action-Value Estimation**:
   $$Q(s, a) = \mathbb{E}[G_t \mid S_t = s, A_t = a]$$
2. **Greedy Action Extraction**:
   $$\pi(s) = \arg\max_a Q(s, a)$$
3. **The Exploration-Exploitation Dilemma**:
   If the policy becomes deterministic too early, many state-action pairs will never be visited, preventing the agent from discovering better trajectories.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Pure Python Blackjack Environment (Sutton & Barto Ch. 5)
# State: (player_sum: 12..21, dealer_showing_card: 1..10, usable_ace: bool)
# Actions: 0 = Stick, 1 = Hit

class BlackjackEnv:
    def __init__(self):
        self.actions = [0, 1]  # 0: Stick, 1: Hit
        
    def _draw_card(self):
        card = np.random.randint(1, 14)
        return min(card, 10)  # Face cards (J, Q, K) have value 10; Ace is 1
        
    def _draw_hand(self):
        return [self._draw_card(), self._draw_card()]
        
    def _get_hand_value(self, hand):
        val = sum(hand)
        has_ace = 1 in hand
        if has_ace and val + 10 <= 21:
            return val + 10, True  # Usable Ace
        return val, False
        
    def reset(self, start_state=None, start_action=None):
        if start_state is not None:
            player_sum, dealer_card, usable_ace = start_state
            self.player_hand = [1, player_sum - 11] if usable_ace else [player_sum // 2, player_sum - player_sum // 2]
            self.dealer_hand = [dealer_card, self._draw_card()]
        else:
            self.player_hand = self._draw_hand()
            while self._get_hand_value(self.player_hand)[0] < 12:
                self.player_hand.append(self._draw_card())
            self.dealer_hand = self._draw_hand()
            
        p_val, usable_ace = self._get_hand_value(self.player_hand)
        return (p_val, self.dealer_hand[0], usable_ace)
        
    def step(self, action):
        if action == 1:  # Hit
            self.player_hand.append(self._draw_card())
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            if p_val > 21:
                return (p_val, self.dealer_hand[0], usable_ace), -1.0, True  # Bust
            return (p_val, self.dealer_hand[0], usable_ace), 0.0, False
        else:  # Stick: Dealer plays
            while True:
                d_val, _ = self._get_hand_value(self.dealer_hand)
                if d_val >= 17:
                    break
                self.dealer_hand.append(self._draw_card())
                
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            d_val, _ = self._get_hand_value(self.dealer_hand)
            
            if d_val > 21 or p_val > d_val:
                reward = 1.0  # Player wins
            elif p_val == d_val:
                reward = 0.0  # Draw
            else:
                reward = -1.0 # Dealer wins
                
            return (p_val, self.dealer_hand[0], usable_ace), reward, True

env = BlackjackEnv()
print("Blackjack environment initialized.")


In [ ]:
# Step 3: Monte Carlo Control Algorithm Framework
def mc_control_framework(env, num_episodes=50000, epsilon=0.1, gamma=1.0):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    N = defaultdict(lambda: np.zeros(len(env.actions)))
    
    def get_action(state, eps):
        if np.random.rand() < eps:
            return np.random.choice(env.actions)
        return np.argmax(Q[state])
        
    for ep in range(1, num_episodes + 1):
        # 1. Generate episode using current policy
        episode = []
        s = env.reset()
        while True:
            a = get_action(s, epsilon)
            next_s, r, done = env.step(a)
            episode.append((s, a, r))
            s = next_s
            if done:
                break
                
        # 2. Evaluate & Update Q(s, a)
        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s_t, a_t, r_t = episode[t]
            G = gamma * G + r_t
            if (s_t, a_t) not in [(step[0], step[1]) for step in episode[:t]]:
                N[s_t][a_t] += 1
                Q[s_t][a_t] += (G - Q[s_t][a_t]) / N[s_t][a_t]
                
    return Q

Q_learned = mc_control_framework(env, num_episodes=60000, epsilon=0.1)
print(f"Total state-action pairs learned: {len(Q_learned)}")


In [ ]:
# Step 4: Extract and Visualize Learned Policy Grid
policy_grid_no_ace = np.zeros((10, 10)) # player 12..21, dealer 1..10

for p_idx, player in enumerate(range(12, 22)):
    for d_idx, dealer in enumerate(range(1, 11)):
        state = (player, dealer, False)
        policy_grid_no_ace[p_idx, d_idx] = np.argmax(Q_learned[state]) # 0 = Stick, 1 = Hit

plt.figure(figsize=(7, 6))
plt.imshow(policy_grid_no_ace, cmap='coolwarm', origin='lower')
plt.colorbar(ticks=[0, 1], label='Action (0: Stick, 1: Hit)')
plt.xticks(range(10), range(1, 11))
plt.yticks(range(10), range(12, 22))
plt.xlabel("Dealer Showing Card")
plt.ylabel("Player Sum")
plt.title("Monte Carlo Control: Learned Policy (No Usable Ace)")
plt.show()


### Summary
- Estimates $Q(s, a)$ directly from returns.
- Alternates policy evaluation and greedy improvement in episodic rounds.
